In [1]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

In [2]:
df = pd.read_csv('homework_6.1.csv')
df

,Z,X,Y
0,0.548814,0,-0.823220
1,0.715189,1,0.842405
2,0.602763,1,0.898618
3,0.544883,0,-0.817325
4,0.423655,0,-0.635482
...,...,...,...
995,0.097676,0,-0.146515
996,0.514922,0,-0.772383
997,0.938412,1,0.730794
998,0.228647,0,-0.342970


Given a dataset, match treated (X = 1) to untreated (X = 0) based on the confounder (Z).

Find the average treatment effect (each item corresponds to one counterfactual) where the counterfactual is the nearest item in the other group (you can use Nearest Neighbors for this.)

Find the average treatment effect on the treated, where each treated item corresponds to a counterfactual untreated item, but we otherwise ignore the untreated items.

Find the average treatment effect on the untreated, where each untreated item corresponds to a counterfactual treated item, but we otherwise ignore the treated items.

Find the optimal treatment effect, which is the maximum treatment effect across all untreated items (i.e., it ends up considering only a single untreated item with its single counterfactual). 

Use the file homework_6.1.csv. 

In [3]:
df_1 = df.copy()
df_x0 = df[df['X'] == 0]
df_x1 = df[df['X'] == 1]

# Given a dataset, match treated (X = 1) to untreated (X = 0) based on the confounder (Z).
nn = NearestNeighbors(n_neighbors=1, algorithm='auto')
nn.fit(df_x0[['Z']]) # Now we have model trained on the 0s. We can get pred for each 1 for the best 0
distances, positional_indices = nn.kneighbors(df_1[['Z']])

# Extract the true original dataframe index of the matches
matched_dataframe_indices = df_x0.index[positional_indices.flatten()]


# Map the matched data back to the X = 1 dataframe
df_1['matched_X0_idx'] = matched_dataframe_indices
df_1['matched_X'] = df_x0.loc[matched_dataframe_indices, 'X'].values
df_1['matched_Y'] = df_x0.loc[matched_dataframe_indices, 'Y'].values
df_1['matched_Z'] = df_x0.loc[matched_dataframe_indices, 'Z'].values
df_1['distance'] = distances.flatten()


df_1[['X', 'Y', 'Z', 'matched_X0_idx', 'matched_X', 'matched_Y', 'matched_Z', 'distance']].head()

# Now we have df containing all the matched pairs. For any row with X=0, it is matched to itself. For any row with X=1 it is matched to an X=0 based on closest Z

,X,Y,Z,matched_X0_idx,matched_X,matched_Y,matched_Z,distance
0,0,-0.823220,0.548814,0,0,-0.823220,0.548814,0.000000
1,1,0.842405,0.715189,93,0,-1.074491,0.716327,0.001138
2,1,0.898618,0.602763,485,0,-0.906177,0.604118,0.001354
3,0,-0.817325,0.544883,3,0,-0.817325,0.544883,0.000000
4,0,-0.635482,0.423655,4,0,-0.635482,0.423655,0.000000


In [4]:
# From text: The difference between the treated sample’s Yand the untreated sample’s Y — where one is real and the other counterfactual — is then averaged over all samples. This gives us the ATE.
# For treated samples (X=1), we use the treated (factual) minus the untreated (counterfactual). For untreated samples, we use the treated (counterfactual) minus the untreated (factual).

In [5]:
# Average Treatment Effect on the Treated (ATT)
df_1_x1 = df_1[df_1['X'] == 1]
att = (df_1_x1['Y']- df_1_x1['matched_Y']).mean()
att

np.float64(1.8464085071912946)

In [6]:
df_1_x0 = df_1[df_1['X'] == 0]
(df_1_x0['Y']- df_1_x0['matched_Y']).mean()

np.float64(0.0)

In [7]:
df_2 = df.copy()
df_x0 = df[df['X'] == 0]
df_x1 = df[df['X'] == 1]

nn = NearestNeighbors(n_neighbors=1, algorithm='auto')
nn.fit(df_x1[['Z']]) # Now we have model trained on the 0s. We can get pred for each 1 for the best 0
distances, positional_indices = nn.kneighbors(df_2[['Z']])

# Extract the true original dataframe index of the matches
matched_dataframe_indices = df_x1.index[positional_indices.flatten()]


# Map the matched data back to the X = 1 dataframe
df_2['matched_X1_idx'] = matched_dataframe_indices
df_2['matched_X'] = df_x1.loc[matched_dataframe_indices, 'X'].values
df_2['matched_Y'] = df_x1.loc[matched_dataframe_indices, 'Y'].values
df_2['matched_Z'] = df_x1.loc[matched_dataframe_indices, 'Z'].values
df_2['distance'] = distances.flatten()


df_2[['X', 'Y', 'Z', 'matched_X1_idx', 'matched_X', 'matched_Y', 'matched_Z', 'distance']].head()

,X,Y,Z,matched_X1_idx,matched_X,matched_Y,matched_Z,distance
0,0,-0.823220,0.548814,751,1,0.925250,0.549501,0.000687
1,1,0.842405,0.715189,1,1,0.842405,0.715189,0.000000
2,1,0.898618,0.602763,2,1,0.898618,0.602763,0.000000
3,0,-0.817325,0.544883,362,1,0.928097,0.543806,0.001077
4,0,-0.635482,0.423655,124,1,0.988072,0.423855,0.000200


In [8]:
df_2_x1 = df_2[df_2['X'] == 1]
(df_2_x1['Y']- df_2_x1['matched_Y']).mean()

np.float64(0.0)

In [9]:
# Average Treatment Effect on the Untreated (ATUT)

df_2_x0 = df_2[df_2['X'] == 0]
atut = (df_2_x0['Y']- df_2_x0['matched_Y']).mean()
atut

np.float64(-1.5494765534751722)

In [10]:
# Average Treatment Effect is the mean of ATT and ATUT
(att + atut) / 2

np.float64(0.14846597685806118)

In [11]:
# Is it equal number of treated and untreated?
len(df_1), len(df_x0), len(df_x1)

(1000, 509, 491)

In [12]:
# Unequal number so we need to weight the result
untreated_adjustment = len(df_x0) / len(df_1)
treated_adjustment = len(df_x1) / len(df_1)
untreated_adjustment, treated_adjustment

(0.509, 0.491)

In [13]:
(treated_adjustment*att + untreated_adjustment*(-atut)) # note sign flip on atut since i had it as negative earlier

np.float64(1.6952701427497883)

Q1. Which is closest to the average treatment effect? 

A1. D (1.695)

Q2. Which is closest to the average treatment effect on the treated? 

A2. D (1.846)

Q3. Which is closest to the average treatment effect on the untreated? 

D (1.549)

Q4. Which is closest to the optimal treatment effect? 

This is the treatment effect for the next sample you’d want to treat. Here, we’re imagining that you are treating samples in a certain order. You rank the samples in terms of importance. The importance of a sample has to do with the benefit of treating it, which is Y(i,treated) - Y(i,untreated).

Assuming you want to get the maximum possible effect, you want to choose the next sample with the maximum treatment effect Y(i,treated) - Y(i,control) taken over all untreated samples i. (There’s no point in considering treated samples; they’re already treated, so they can’t be the “next” samples to be treated.) 

In [14]:
# fit on treated pool
nn_treated = NearestNeighbors(n_neighbors=1, algorithm='auto')
nn_treated.fit(df_x1[['Z']])

# find closest treated row for each untreated
distances, positional_indices = nn_treated.kneighbors(df_x0[['Z']])
matched_treated_indices = df_x1.index[positional_indices.flatten()]

# extract the counterfactual Y(1) for the untreated individuals
df_x0['estimated_Y_treated'] = df_x1.loc[matched_treated_indices, 'Y'].values

# calculate Y(i,treated) - Y(i,untreated)
df_x0['individual_treatment_effect'] = df_x0['estimated_Y_treated'] - df_x0['Y']

# find the next optimal treatment effect (the absolute maximum)
optimal_next_effect = df_x0['individual_treatment_effect'].max()
optimal_next_sample_idx = df_x0['individual_treatment_effect'].idxmax()

optimal_next_effect, optimal_next_sample_idx


C:\Users\caleb\AppData\Local\Temp\ipykernel_26088\394820989.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_x0['estimated_Y_treated'] = df_x1.loc[matched_treated_indices, 'Y'].values
C:\Users\caleb\AppData\Local\Temp\ipykernel_26088\394820989.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_x0['individual_treatment_effect'] = df_x0['estimated_Y_treated'] - df_x0['Y']


(np.float64(2.172469885577008), np.int64(298))

Reflection 6 Question 1

In [18]:
df_2

,Z,X,Y,matched_X1_idx,matched_X,matched_Y,matched_Z,distance
0,0.548814,0,-0.823220,751,1,0.925250,0.549501,0.000687
1,0.715189,1,0.842405,1,1,0.842405,0.715189,0.000000
2,0.602763,1,0.898618,2,1,0.898618,0.602763,0.000000
3,0.544883,0,-0.817325,362,1,0.928097,0.543806,0.001077
4,0.423655,0,-0.635482,124,1,0.988072,0.423855,0.000200
...,...,...,...,...,...,...,...,...
995,0.097676,0,-0.146515,551,1,1.151379,0.097243,0.000433
996,0.514922,0,-0.772383,849,1,0.942181,0.515638,0.000716
997,0.938412,1,0.730794,997,1,0.730794,0.938412,0.000000
998,0.228647,0,-0.342970,941,1,1.086319,0.227362,0.001284


In [21]:
import numpy as np


np.percentile(
    (df_2_x0['Y']- df_2_x0['matched_Y']),
    90
)


np.float64(-1.2593238427299611)